In [1]:
import import_ipynb
import pandas_ta as ta
import pandas as pd
import numpy as np

In [2]:
PARAMS = {
    "ROC_direction": 20,
    "SMA_Trend_structure": [20,50],
    "MACD_Trend_acceleration": 9,
    "Level_ZScore_Market_tension": 20,
    "RSI_Momentum_exhaustion": 14,
    "Ret_ZScore_Price_shock": 7,
    "Volatility_Market_risk": 14,
    "Volatility_Structure": 20,
    "Liquidity_Pressure": 7
}

In [ ]:
class Technical_Indicators():
    #Trend
    def feature_roc(self, series, length=20):
        roc = ta.roc(series, length=length)
        return roc.rename("ROC_direction")

    def feature_roc_2(self, series, length, std_coef = (-0.5, 0.4)):
        returns = series.pct_change()
        neg_std_coef, pos_std_coef = std_coef
        rolling_std = returns.rolling(window=length).std()
        rolling_neg_std_effect = rolling_std * neg_std_coef
        rolling_pos_std_effect = rolling_std * pos_std_coef
        return rolling_neg_std_effect, rolling_pos_std_effect

    #Trend structure
    def feature_sma_structure(self, series, short=20, long=50):
        sma_short = ta.sma(series, length=short)
        sma_long = ta.sma(series, length=long)
        structure = sma_short - sma_long
        return structure.rename("SMA_Trend_structure")

    def feature_sma_2(self, series, short=10, long=50):
        s_short = series.rolling(short).mean()
        s_long = series.rolling(long).mean()
        sma_diff = (s_short - s_long) / s_long
        roll_mean = sma_diff.rolling(long).mean()
        roll_std = sma_diff.rolling(long).std().replace(0, np.nan)
        z = (sma_diff - roll_mean) / roll_std
        return s_short, s_long, z, sma_diff

    #Trend acceleration
    def feature_macd(self, series, fast=12, slow=26, signal=9):
        macd = ta.macd(series, fast=fast, slow=slow, signal=signal)
        hist = macd[f"MACDh_{fast}_{slow}_{signal}"]
        return hist.rename("MACD_Trend_acceleration")

    def feature_macd_2(self, series, fast=12, slow=26, signal=9):
        ema_fast = series.ewm(span=fast, adjust=False).mean()
        ema_slow = series.ewm(span=slow, adjust=False).mean()
        macd = ema_fast - ema_slow
        macd_signal = macd.ewm(span=signal, adjust=False).mean()
        macd_hist = macd - macd_signal
        return macd, macd_signal, macd_hist

    #Mean reversion
    def feature_level_zscore(self, series, length=20):
        rolling_mean = series.rolling(window=length).mean()
        rolling_std = series.rolling(window=length).std()
        z = (series - rolling_mean) / rolling_std
        return z.rename("Level_ZScore_Market_tension")

    #Momentum exhaustion
    def feature_rsi(self, series, length=14):
        rsi = ta.rsi(series, length=length)
        return rsi.rename("RSI_Momentum_exhaustion")

    def feature_rsi_2(self, series, length):
        delta = series.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=length).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=length).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi.rename("RSI_Momentum_exhaustion_2")

    #Shock
    def feature_return_zscore(self, series, length=20):
        returns = series.pct_change()
        rolling_std = returns.rolling(window=length).std()
        z = returns / rolling_std
        return z.rename("Ret_ZScore_Price_shock")

    #Volatility level
    def feature_volatility(self, series, length=20):
        returns = series.pct_change()
        vol = returns.rolling(window=length).std()
        return vol.rename("Volatility_Market_risk")

    def feature_volatility_2(self, series, span):
        prev_day_start = series.close.index.searchsorted(series.close.index - pd.Timedelta(days=1))
        prev_day_start = prev_day_start[prev_day_start > 0]
        prev_day_start = pd.Series(series.close.index[prev_day_start - 1], index=series.close.index[series.close.shape[0] - prev_day_start.shape[0]:])
        daily_returns = series.close.loc[prev_day_start.index] / series.close.loc[prev_day_start.values].values - 1
        vol = daily_returns.ewm(span=span).std()
        return vol.rename("Volatility_Market_risk_2")

    #Volatility structure
    def feature_bollinger_width(self, series, length=20):
        ma = series.rolling(length).mean()
        std = series.rolling(length).std()
        upper = ma + 2 * std
        lower = ma - 2 * std
        width = (upper - lower) / ma   # normalizacja
        return width.rename("Volatility_Structure")

    #Liquidity pressure
    def feature_volume_pressure(self, volume, length=20):
        vol_z = (volume - volume.rolling(length).mean()) / volume.rolling(length).std()
        return vol_z.rename("Liquidity_Pressure")

    def build_market_features(self, close, volume, params=PARAMS):
        features = []
        if "ROC_direction" in params:
            features.append(self.feature_roc(close, length=params['ROC_direction']))
        if "RSI_Momentum_exhaustion" in params:
            features.append(self.feature_rsi(close, length=params['RSI_Momentum_exhaustion']))
        if "SMA_Trend_structure" in params:
            features.append(self.feature_sma_structure(close, short=params['SMA_Trend_structure'][0], long=params['SMA_Trend_structure'][1])),
        if "MACD_Trend_acceleration" in params:
            features.append(self.feature_macd(close, signal=params['MACD_Trend_acceleration']))
        if "Level_ZScore_Market_tension" in params:
            features.append(self.feature_level_zscore(close, length=params['Level_ZScore_Market_tension'])),
        if "Ret_ZScore_Price_shock" in params:
            features.append(self.feature_return_zscore(close, length=params['Ret_ZScore_Price_shock']))
        if "Volatility_Market_risk" in params:
            features.append(self.feature_volatility(close, length=params['Volatility_Market_risk']))
        if "Volatility_Structure" in params:
            features.append(self.feature_bollinger_width(close, length=params['Volatility_Structure']))
        if "Liquidity_Pressure" in params:
            features.append(self.feature_volume_pressure(volume, length=params['Liquidity_Pressure']))
        df = pd.concat(features, axis=1)
        return df
